In [1]:
import sys
sys.path.insert(0, '..')
from model.data import getMyData
from model.embedding import DataEmbedding
from model.encoder import Encoder,ConvLayer
from model.attention import AttentionLayer,ProbAttention

import torch_ep.embed as epEmbed
import torch_ep.encoder as epEncoder
import torch

import numpy  as np
import tensorflow as tf
import math

from tensorflow.keras.layers import Dense
from keras.backend import softmax

import matplotlib.pyplot as plt
import seaborn as sns
import seaborn.objects as so

In [ ]:
#Data Standard sinusoide
DATA_SIZE = 3000 #TTFData.shape[0]
SEQ_LEN = 20
BATCH_SIZE = 32
myDataFct = getMyData(data_size=DATA_SIZE, seq_len=SEQ_LEN,batch_len=BATCH_SIZE)
dataset, datasetVal = myDataFct.get()
myDataFct.display()
btX = (list(enumerate(myDataFct.dataset))[0][1][0])
btY = (list(enumerate(myDataFct.dataset))[0][1][1])
btX_T = torch.from_numpy(btX.numpy())
btY_T = torch.from_numpy(btY.numpy())

In [2]:
# Get Features from ETTh1
import pandas as pd
import os
import re
root_path = "../../data/"
data_path = "ETTh1.csv"
df_raw = pd.read_csv(os.path.join(root_path,data_path))

SEQ_LEN = 96
BATCH_SIZE = 32
GLOBAL_SIZE=1000

# split = re.compile('-|:| ')
# featuresDate = (df_raw['date'].str.split(split, expand=True)).astype(np.int64).iloc[:,:4]
featuresData = df_raw.iloc[:,1:8].astype(np.float32)

# On ajoute les infos temporelles : mois/jour/joursemain/heure
df_raw['mois'] = df_raw.date.apply(lambda row:int(row[5:7])).astype(np.int64)
df_raw['jour'] = df_raw.date.apply(lambda row:int(row[8:10])).astype(np.int64)
df_raw['heure'] = df_raw.date.apply(lambda row:int(row[11:13])).astype(np.int64)
df_raw['sJour'] = df_raw['date'].astype('datetime64[s]').dt.dayofweek.astype(np.int64)
featuresDate = df_raw[['mois','jour','sJour','heure']].astype(np.int64)
# featuresDate.describe()

# Convert Features to numpy array
featuresData = featuresData.to_numpy()
featuresDate = featuresDate.to_numpy()
# Create a dataset Feature Data
X = np.array([featuresData[i:i+SEQ_LEN] for i in range(0, featuresData.shape[0]-SEQ_LEN)], dtype=np.float32)
Y = np.array([featuresData[i+SEQ_LEN] for i in range(0, featuresData.shape[0]-SEQ_LEN-1)], dtype=np.float32)
Xt = tf.convert_to_tensor(X[:GLOBAL_SIZE,:,:], dtype=tf.float32)
XT = torch.from_numpy(X[:GLOBAL_SIZE,:,:])

# Create a dataset : Features Date
X = np.array([featuresDate[i:i+SEQ_LEN] for i in range(0, featuresDate.shape[0]-SEQ_LEN)], dtype=np.float32)
Y = np.array([featuresDate[i+SEQ_LEN] for i in range(0, featuresDate.shape[0]-SEQ_LEN-1)], dtype=np.float32)
XtDate = tf.convert_to_tensor(X[:GLOBAL_SIZE,:,:], dtype=tf.float32)
XTDate = torch.from_numpy(X[:GLOBAL_SIZE,:,:])

2024-09-01 20:47:45.440469: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2024-09-01 20:47:45.440490: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2024-09-01 20:47:45.440497: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2024-09-01 20:47:45.440533: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2024-09-01 20:47:45.440547: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [ ]:
#Final Embedding
d_model = 512
RATE = 0.05
fig = plt.figure(figsize=(18,2))
sns.lineplot(x=df_raw['date'][:GLOBAL_SIZE],y=df_raw['OT'][:GLOBAL_SIZE]).set_title('Torch EMbedding');
#Embedding Torch
dataTEmb = epEmbed.DataEmbedding(c_in=7,d_model=d_model,dropout=RATE)
d = dataTEmb(XT, XTDate)
fig = plt.figure(figsize=(18,2))
sns.heatmap(d.detach()[0], vmin=-1, vmax=1, cmap=sns.color_palette("hls", 256)).set_title('Torch EMbedding');
#Embedding Keras
dataKEmb = DataEmbedding(seq_len=SEQ_LEN, d_model=d_model,rate=RATE)
Kdk = dataKEmb(x=Xt, x_mark=XtDate, training=True)
fig = plt.figure(figsize=(18,2))
sns.heatmap(Kdk[0], vmin=-1, vmax=1, cmap=sns.color_palette("hls", 256)).set_title('Keras Embedding');

In [ ]:
def _prob_QK( Q, K, sample_k, n_top): # n_top: c*ln(L_q)
    # Q [B, H, L, D] /batch/seq_len/head/reste...
    B, H, L_K, E = K.shape
    _, _, L_Q, _ = Q.shape

    print('dim: ', B,H,L_K,L_Q,E)

    # calculate the sampled Q_K
    K_torch = torch.from_numpy(K.numpy())
    K_torch_expand = K_torch.unsqueeze(-3).expand(B, H, L_Q, L_K, E)
    K_expand = tf.broadcast_to(tf.expand_dims(K,axis=2), (B, H, L_Q, L_K, E))

    # index_sample_torch = torch.randint(L_K, (L_Q, sample_k)) # real U = U_part(factor*ln(L_k))*L_q
    index_sample = tf.random.uniform(maxval=L_K, shape=(L_Q, sample_k),dtype=tf.dtypes.int64)
    index_sample_torch = torch.from_numpy(index_sample.numpy()) # real U = U_part(factor*ln(L_k))*L_q

    # print(torch.arange(L_Q).unsqueeze(1).shape, index_sample_torch.shape)
    # print(tf.expand_dims(np.arange(L_Q),1).shape, index_sample.shape)
    K_sample_torch = K_torch_expand[:, :, torch.arange(L_Q).unsqueeze(1), index_sample_torch, :]
    # K_sample = K_expand[:, :, tf.expand_dims(np.arange(L_Q),1), index_sample, :]
    v = tf.expand_dims(np.arange(L_Q),1)
    K_sample = tf.convert_to_tensor(K_expand.numpy()[:, :, v.numpy(), index_sample.numpy(), :])
    # K_sample = tf.convert_to_tensor(K_sample_torch.numpy())
    print(K_sample_torch.shape, K_sample.shape)
    
    QT = torch.from_numpy(Q.numpy())
    Q_K_sample_torch = torch.matmul(QT.unsqueeze(-2), K_sample_torch.transpose(-2, -1)).squeeze(-2)
    Q_K_sample = tf.squeeze(tf.matmul(tf.expand_dims(Q,axis=-2),tf.transpose(K_sample,perm=(0,1,2,4,3))),axis=-2)

    # find the Top_k query with sparisty measurement
    MT = Q_K_sample_torch.max(-1)[0] - torch.div(Q_K_sample_torch.sum(-1), L_K)
    print('Reduce Torch', Q_K_sample_torch.max(-1)[0].shape)
    print('MT',MT.shape)
    print('Q_K sample', Q_K_sample.shape)
    print('REDUCE',tf.reduce_max(Q_K_sample, axis=-1).shape)
    M = tf.reduce_max(Q_K_sample, axis=-1) - tf.divide(tf.reduce_sum(Q_K_sample, axis=-1), L_K)
    print('M Keras', M.shape)
    
    MT_top = MT.topk(n_top, sorted=False)[1]
    print('MT_top',MT_top.shape)
    M_top = tf.math.top_k(M, k=n_top, sorted=False)[1]
    print('M_top',M_top.shape)

    # use the reduced Q to calculate Q_K
    QT_reduce = QT[torch.arange(B)[:, None, None],
                    torch.arange(H)[None, :, None],
                    MT_top, :] # factor*ln(L_q)
    Q_reduce = Q.numpy()[np.arange(B)[:, None, None],
                    np.arange(H)[None, :, None],
                    M_top.numpy(), :] # factor*ln(L_q)
    # Q_reduce = tf.convert_to_tensor(QT_reduce.numpy())
    print('Q_reduce', Q_reduce.shape)
    KT= torch.from_numpy(K.numpy())
    Q_KT = torch.matmul(QT_reduce, KT.transpose(-2, -1)) # factor*ln(L_q)*L_k
    Q_K = tf.matmul(Q_reduce, tf.transpose(K, perm=(0,1,3,2))) # factor*ln(L_q)*L_k

    return Q_KT, MT_top, Q_K, M_top
    # return Q_K, M_top


In [ ]:
x = tf.convert_to_tensor(np.array([[1, 2, 3, 4],[5,6,7,8],[9,10,11,12]]))
# x = torch.tensor([[1, 2, 3, 4],[5,6,7,8],[9,10,11,12]])
xT = torch.from_numpy(x.numpy())
m = tf.random.uniform(maxval=3, shape=(3, 4),dtype=tf.dtypes.int64)
mT= torch.from_numpy(m.numpy()) # real U = U_part(factor*ln(L_k))*L_q
vT =  torch.arange(3).unsqueeze(1)
v = tf.convert_to_tensor(vT.numpy())
print('v',v.numpy())
print('m',m.numpy())
print(x)
K_sample_torch = xT[vT, mT]
print('torch',K_sample_torch)
# test gather_nd

xN = x.numpy()
mN = m.numpy()
vN = v.numpy()
K = xN[vN,mN]
# K = tf.gather_nd(x,indices=m)
# K = x[v,m]
print('K',K)


# print(m,mT,x)


In [ ]:
def _get_initial_context( V, L_Q):
    B, H, L_V, D = V.shape
    VT = torch.from_numpy(V.numpy())
    # if not self.mask_flag:
    VT_sum = VT.mean(dim=-2)
    V_sum = tf.math.reduce_mean(V, axis=-2)
    contextT = VT_sum.unsqueeze(-2).expand(B, H, L_Q, V_sum.shape[-1]).clone()
    contex = tf.broadcast_to(tf.expand_dims(V_sum,axis=-2), (B, H, L_Q, V_sum.shape[-1])) # .clone()
    # else: # use mask
    #     assert(L_Q == L_V) # requires that L_Q == L_V, i.e. for self-attention only
    #     contex = V.cumsum(dim=-2)
    return contex, contextT

In [ ]:
def _update_context( context_in, V, scores, index, a, b, L_Q, attn_mask):
    B, H, L_V, D = V.shape # Batch/Head/Len_Seq / reste ...

    # if self.mask_flag:
    #     attn_mask = ProbMask(B, H, L_Q, index, scores, device=V.device)
    #     scores.masked_fill_(attn_mask.mask, -np.inf)

    attnT = torch.softmax(a, dim=-1) # nn.Softmax(dim=-1)(scores)
    attn = tf.nn.softmax(scores, axis=-1)

    print('Attn Torch', attnT.shape)
    print('Attn', attn.shape)

    contextT = torch.from_numpy(context_in.numpy())
    VT = torch.from_numpy(V.numpy())
    contextT[torch.arange(B)[:, None, None],
                torch.arange(H)[None, :, None],
                b, :] = torch.matmul(attnT, VT).type_as(contextT)
    print('context Torch',contextT.shape)

    context_np = context_in.numpy()
    context_np[np.arange(B)[:, None, None],
                np.arange(H)[None, :, None],
                index.numpy(), :] = (tf.matmul(attn, V).numpy())

    print('context',context_in.shape,context_in.dtype)
    
    # if self.output_attention:
    #     attns = (torch.ones([B, H, L_V, L_V])/L_V).type_as(attn).to(attn.device)
    #     attns[torch.arange(B)[:, None, None], torch.arange(H)[None, :, None], index, :] = attn
        # return (context_in, attns)
    # else:
    return (tf.convert_to_tensor(context_np), contextT,  None)

In [ ]:
#attention try
HEAD=8
FACTOR = 5
proj = Dense(d_model*3)
dec_emb = tf.reshape(proj(tf.slice(Kdk,[0,0,0],[32,SEQ_LEN,d_model])),[BATCH_SIZE,SEQ_LEN,HEAD,3*d_model//HEAD])
# dec_emb = tf.transpose(dec_emb,perm=[0,2,1,3])
print(dec_emb.shape)
q = tf.slice(dec_emb,[0,0,0,0],[32,SEQ_LEN,8,64])
k = tf.slice(dec_emb,[0,0,0,64],[32,SEQ_LEN,8,64])
v = tf.slice(dec_emb,[0,0,0,128],[32,SEQ_LEN,8,64])
B, L_Q, H, D = q.shape # Batch/Seq_len/Head/reste...
_, L_K, _, _ = k.shape

q = tf.transpose(q, (0,2,1,3))
k = tf.transpose(k, (0,2,1,3))
v = tf.transpose(v, (0,2,1,3))

U_part = FACTOR * np.ceil(np.log(L_K)).astype('int').item() # c*ln(L_k)
u = FACTOR * np.ceil(np.log(L_Q)).astype('int').item() # c*ln(L_q) 

U_part = U_part if U_part<L_K else L_K
u = u if u<L_Q else L_Q

a, b, scores_top, index = _prob_QK(q, k, sample_k=U_part, n_top=u)

print(scores_top.shape, index.shape)
print(a.shape, b.shape)

# add scale factor
scale = 1./math.sqrt(D)
if scale is not None:
    scores_top = scores_top * scale
# get the context
context, contextT = _get_initial_context(v, L_Q)
print('contextT', contextT.shape)
print('Context', context.shape)
fig = plt.figure(figsize=(18,2))
sns.heatmap(contextT[0,0], vmin=-10, vmax=10, cmap=sns.color_palette("hls", 256)).set_title('Context Avant');

# update the context with selected top_k queries
context, contextT, attn = _update_context(context, v, scores_top, index, a, b, L_Q, None)

print(contextT[0,0,0], contextT.shape)
print(context[0,0,0],context.shape)

# return context.transpose(2,1).contiguous(), attn

fig = plt.figure(figsize=(18,2))
sns.heatmap(contextT[0,0,:,:], vmin=-10, vmax=10, cmap=sns.color_palette("hls", 256)).set_title('Context ');
fig = plt.figure(figsize=(18,2))
sns.heatmap(context[0,0,:,:], vmin=-10, vmax=10, cmap=sns.color_palette("hls", 256)).set_title('Context ');


In [3]:
# Final Test with AttentionLayer
d_model = 512
HEAD = 8
RATE = 0.05
SEQ_LEN = 96
FACTOR = 5

#Embedding
encEmb = DataEmbedding(seq_len=SEQ_LEN, d_model=d_model,rate=RATE)
x = encEmb(x=Xt, x_mark=XtDate, training=True)
print(x.shape)

attLayer = AttentionLayer(ProbAttention(False,FACTOR,None,RATE,False),d_model=d_model,heads=HEAD)
newX, attn = attLayer(x,x,x,None)
print(newX.shape, attn.shape)

(1000, 96, 512)
(1000, 8, 96, 64)
(1000, 8, 96, 1, 64)


ResourceExhaustedError: Exception encountered when calling layer 'prob_attention' (type ProbAttention).

{{function_node __wrapped__BroadcastTo_device_/job:localhost/replica:0/task:0/device:GPU:0}} OOM when allocating tensor with shape[1000,8,96,96,64] and type float on /job:localhost/replica:0/task:0/device:GPU:0 by allocator Simple allocator [Op:BroadcastTo]

Call arguments received by layer 'prob_attention' (type ProbAttention):
  • q=tf.Tensor(shape=(1000, 96, 8, 64), dtype=float32)
  • k=tf.Tensor(shape=(1000, 96, 8, 64), dtype=float32)
  • v=tf.Tensor(shape=(1000, 96, 8, 64), dtype=float32)
  • attn_mask=None